In [ ]:

import pandas as pd

# 读取数据
file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv'
data = pd.read_csv(file_path)

# 查看数据前几行
print(data.head())

# 查看数据的基本信息
print(data.info())

# 查看数据的描述性统计
print(data.describe())


     id  allelectrons_Total  ...  density_Average  Hardness
0  2124                30.0  ...          0.51006       6.0
1   394                64.0  ...          4.74000       3.3
2  3101                97.0  ...          1.79976       5.3
3  1737               151.0  ...          7.77500       1.8
4   561               131.0  ...          1.92652       5.5

[5 rows x 13 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8325 non-null   int64  
 1   allelectrons_Total     8325 non-null   float64
 2   density_Total          8325 non-null   float64
 3   allelectrons_Average   8325 non-null   float64
 4   val_e_Average          8325 non-null   float64
 5   atomicweight_Average   8325 non-null   float64
 6   ionenergy_Average      8325 non-null   float64
 7   el_neg_chi_Average     8325 non-null 

In [ ]:


from sklearn.preprocessing import StandardScaler

# 分离特征和目标变量
X = data.drop(columns=['id', 'Hardness'])
y = data['Hardness']

# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 查看标准化后的前几行数据
print(pd.DataFrame(X_scaled, columns=X.columns).head())



   allelectrons_Total  density_Total  ...  zaratio_Average  density_Average
0           -0.449761      -0.774773  ...        -0.204818        -0.836903
1           -0.293378      -0.068134  ...        -0.301943         1.334983
2           -0.141595      -0.062591  ...        -0.105408        -0.174700
3            0.106778       0.514155  ...        -1.221058         2.893320
4            0.014788       0.585200  ...        -0.311987        -0.109614

[5 rows x 11 columns]


In [ ]:



import numpy as np

# 计算特征与目标变量的相关性
correlations = X.corrwith(y)
print(correlations)

# 选择相关性高于某个阈值的特征
threshold = 0.1
selected_features = correlations[abs(correlations) > threshold].index
print(f"Selected features: {selected_features}")

# 选择标准化后的特征
X_selected = X_scaled[:, X.columns.isin(selected_features)]

# 将选择的特征转换为DataFrame
X_selected_df = pd.DataFrame(X_selected, columns=selected_features)
print(X_selected_df.head())


allelectrons_Total      -0.064286
density_Total           -0.162258
allelectrons_Average    -0.406722
val_e_Average            0.155745
atomicweight_Average    -0.408736
ionenergy_Average        0.204471
el_neg_chi_Average       0.288193
R_vdw_element_Average   -0.064431
R_cov_element_Average   -0.188326
zaratio_Average          0.054903
density_Average         -0.366399
dtype: float64
Selected features: Index(['density_Total', 'allelectrons_Average', 'val_e_Average',
       'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average',
       'R_cov_element_Average', 'density_Average'],
      dtype='object')
   density_Total  allelectrons_Average  ...  R_cov_element_Average  density_Average
0      -0.774773             -0.672137  ...              -0.435357        -0.836903
1      -0.068134              0.566681  ...               1.552238         1.334983
2      -0.062591             -0.469638  ...               0.042074        -0.174700
3       0.514155              3.171377  ...

In [ ]:


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import median_absolute_error

# 分割数据集为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

# 初始化随机森林回归模型
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 预测测试集
y_pred = model.predict(X_test)

# 计算中位绝对误差
mae = median_absolute_error(y_test, y_pred)
print(f"Median Absolute Error (MAE): {mae:.4f}")


Median Absolute Error (MAE): 0.6336
